# 01 - Parallel VLM Inference to Database

**Features:**
1. **Skip existing samples** - Check DB before running inference
2. **ThreadPoolExecutor** - Parallel processing (avoids pickling issues in notebooks)
3. **Batch inserts** - Accumulate records and insert in batches
4. **tqdm progress** - Visual progress tracking

In [18]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import os
import time
import hashlib
import traceback
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple, Set
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm
from sqlalchemy import text

print('Imports successful!')

Imports successful!


In [19]:
@dataclass
class InferenceConfig:
    run_id: str = field(default_factory=lambda: f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    models_yaml: str = '../configs/models.yaml'
    gpu_endpoints: Dict[str, str] = field(default_factory=lambda: {
        'deepseek_ocr': 'http://localhost:9002/metrics',
        'qwen2_5_vl_3b': 'http://localhost:9001/metrics',
        'qwen2_5_vl_7b': 'http://localhost:9001/metrics',
        'qwen3_vl_8b_thinking': 'http://localhost:9000/metrics',
        'gemma_3_27b': 'http://localhost:9000/metrics',
    })
    samples_per_config: int = 50
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    temperature: float = 0.0
    max_tokens: int = 512
    top_p: float = 1.0
    
    # Parallelization (ThreadPool instead of ProcessPool)
    max_config_workers: int = 4  # Configs processed in parallel
    batch_insert_size: int = 50
    
    # Skip existing
    skip_existing: bool = True  # Set False to re-run everything

config = InferenceConfig()
print(f'Run ID: {config.run_id}')
print(f'Skip existing: {config.skip_existing}')

Run ID: run_20251206_132947
Skip existing: True


In [20]:
# Test imports and DB connection
from ares.db.connection import get_engine, test_connection
from ares.db.operations import (
    insert_samples, insert_images, insert_responses,
    get_existing_responses, get_existing_sample_ids
)
from ares.configs.db_config import TABLES, MODEL_NAMES

if not test_connection():
    raise RuntimeError('DB connection failed!')
print('DB connected!')

✓ Connected to database. Server time: 2025-12-06 18:29:47.575167+00:00
DB connected!


In [21]:
# Initialize clients (shared across threads)
from ares.inference_api_call.client import WhichVLMClient
from ares.evaluation.sample_processor import process_sample_normalized
from ares.utils.common_utils import return_model_specs
from ares.data.dataset_loader import CauldronLoader
from ares.evaluation.evaluation import Scorer
from ares.metrics.metrics_client import GPUMetricsClient

vlm_client = WhichVLMClient.from_yaml(config.models_yaml)
gpu_client = GPUMetricsClient(endpoints=config.gpu_endpoints)
model_specs = return_model_specs()
scorer = Scorer()

print('Clients initialized!')

Configured models:
  id=0 name=deepseek_ocr prefix=deepseek_ocr__
  id=1 name=qwen2_5_vl_3b prefix=qwen2_5_vl_3b__
  id=2 name=qwen2_5_vl_7b prefix=qwen2_5_vl_7b__
  id=3 name=qwen3_vl_8b_thinking prefix=qwen3_vl_8b_thinking__
  id=4 name=gemma_3_27b prefix=gemma_3_27b__
Clients initialized!


## Check Existing Data

In [22]:
# Show current DB state
engine = get_engine()
with engine.connect() as conn:
    samples = conn.execute(text('SELECT COUNT(*) FROM vlm_samples')).scalar()
    responses = conn.execute(text('SELECT COUNT(*) FROM vlm_responses')).scalar()

print(f'Current DB state:')
print(f'  Samples: {samples:,}')
print(f'  Responses: {responses:,}')

Current DB state:
  Samples: 24,548
  Responses: 122,740


In [23]:
# Get list of configs to process
from ares.configs.config import ALL_CAULDRON_CONFIGS

# TEST MODE: Small subset
# configs_to_process = ['ai2d', 'aokvqa', 'chartqa']  

# PRODUCTION: All configs
configs_to_process = ALL_CAULDRON_CONFIGS

print(f'Configs to process: {len(configs_to_process)}')

Configs to process: 50


In [24]:
def process_single_config(source_config: str) -> Dict[str, Any]:
    '''
    Process all samples from one Cauldron config.
    Uses ThreadPoolExecutor (works in notebooks, no pickling issues).
    
    Checks existing responses and skips models already processed.
    '''
    results = {'config': source_config, 'processed': 0, 'skipped': 0, 'errors': 0}
    
    try:
        samples = CauldronLoader.load_samples(
            source_config, 
            n_samples=config.samples_per_config, 
            random_sample=True
        )
    except Exception as e:
        results['error'] = str(e)
        return results
    
    # Generate sample_ids for these samples
    sample_ids = []
    for idx, sample in enumerate(samples):
        hash_input = f"{source_config}_{idx}_{sample.get('question', '')[:50]}"
        hash_val = hashlib.md5(hash_input.encode()).hexdigest()[:8]
        sample_ids.append(f"{source_config}_{idx}_{hash_val}")
    
    # Check what's already in DB
    existing_responses = {}
    if config.skip_existing:
        existing_responses = get_existing_responses(sample_ids)
        fully_complete = sum(1 for sid in sample_ids if len(existing_responses.get(sid, set())) >= 5)
        results['skipped'] = fully_complete
    
    # Batch accumulators
    sample_batch, image_batch, response_batch = [], [], []
    
    for idx, sample in enumerate(samples):
        sample_id = sample_ids[idx]
        
        # Check which models we need to run
        existing_models = existing_responses.get(sample_id, set())
        if config.skip_existing and len(existing_models) >= 5:
            continue  # All 5 models already done, skip entirely
        
        # Determine which models to run
        models_to_run = [m for m in MODEL_NAMES if m not in existing_models] if config.skip_existing else None
        
        try:
            result = process_sample_normalized(
                sample=sample,
                sample_idx=idx,
                source_config=source_config,
                vlm_client=vlm_client,
                gpu_client=gpu_client,
                scorer=scorer,
                config=config,
                model_specs=model_specs,
                models_to_run=models_to_run,
            )
            
            if result:
                sample_record, image_record, response_records = result
                sample_batch.append(sample_record)
                image_batch.append(image_record)
                response_batch.extend(response_records)
                results['processed'] += 1
                
                # Batch insert
                if len(sample_batch) >= config.batch_insert_size:
                    insert_images(image_batch)
                    insert_samples(sample_batch)
                    insert_responses(response_batch)
                    sample_batch, image_batch, response_batch = [], [], []
        except Exception as e:
            results['errors'] += 1
            print(f'  Error in {source_config} sample {idx}: {e}')
    
    # Final batch insert
    if sample_batch:
        insert_images(image_batch)
        insert_samples(sample_batch)
        insert_responses(response_batch)
    
    return results

In [ ]:
# Run with ThreadPoolExecutor (no pickling issues!)
print(f'Starting parallel processing...')
print(f'  Workers: {config.max_config_workers}')
print(f'  Configs: {len(configs_to_process)}')
print(f'  Skip existing: {config.skip_existing}')
print()

start_time = time.time()
all_results = []

with ThreadPoolExecutor(max_workers=config.max_config_workers) as executor:
    futures = {executor.submit(process_single_config, cfg): cfg for cfg in configs_to_process}
    
    with tqdm(total=len(futures), desc='Configs') as pbar:
        for future in as_completed(futures):
            config_name = futures[future]
            try:
                result = future.result()
                all_results.append(result)
                pbar.set_postfix({
                    'config': config_name[:15], 
                    'done': result.get('processed', 0),
                    'skip': result.get('skipped', 0)
                })
            except Exception as e:
                all_results.append({'config': config_name, 'error': str(e)})
                print(f'Error in {config_name}: {e}')
            pbar.update(1)

elapsed = time.time() - start_time
print(f'\nCompleted in {elapsed:.1f}s')

Starting parallel processing...
  Workers: 4
  Configs: 50
  Skip existing: True



Configs:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

In [ ]:
# Summary
print('='*60)
print('RESULTS SUMMARY')
print('='*60)

total_processed = sum(r.get('processed', 0) for r in all_results)
total_skipped = sum(r.get('skipped', 0) for r in all_results)
total_errors = sum(r.get('errors', 0) for r in all_results)

print(f'Total samples processed: {total_processed}')
print(f'Total samples skipped (already in DB): {total_skipped}')
print(f'Total errors: {total_errors}')
if elapsed > 0 and total_processed > 0:
    print(f'Throughput: {total_processed / elapsed:.1f} samples/sec')

# Show errors if any
errors = [r for r in all_results if 'error' in r]
if errors:
    print(f'\nConfigs with errors:')
    for e in errors[:5]:
        print(f"  {e['config']}: {e['error'][:50]}")

In [ ]:
# Results table
df_results = pd.DataFrame(all_results)
df_results = df_results.sort_values('processed', ascending=False)
df_results

In [ ]:
# Verify DB
engine = get_engine()
with engine.connect() as conn:
    for name, table in TABLES.items():
        result = conn.execute(text(f'SELECT COUNT(*) FROM {table}'))
        print(f'{name}: {result.scalar():,} rows')

---
## Alternative: Sequential (for debugging)

In [ ]:
# Sequential version (useful for debugging)
def run_sequential(configs, skip_existing=True):
    '''Process configs one by one (for debugging).'''
    config.skip_existing = skip_existing
    
    for cfg in tqdm(configs, desc='Sequential'):
        result = process_single_config(cfg)
        print(f"{cfg}: processed={result.get('processed', 0)}, skipped={result.get('skipped', 0)}")

# Usage:
# run_sequential(['ai2d', 'chartqa'], skip_existing=True)